In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
x_train = pd.read_csv(Path.cwd().parent / "data/german_credit_train.csv")
x_test = pd.read_csv(Path.cwd().parent / "data/german_credit_test.csv")


In [5]:
df = x_train.copy()

In [6]:
y_sample = df["Risk"]
x_train = df.drop(columns="Risk", axis=1)

In [7]:
X_sample_num = x_train.select_dtypes(include = np.number)
X_sample_cat = x_train.select_dtypes(exclude = np.number)

ordinal_cols = X_sample_cat.columns
scale_cols = X_sample_num.columns

In [8]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

scaler = StandardScaler()
ordinal = OrdinalEncoder()

x_train[scale_cols] = scaler.fit_transform(x_train[scale_cols])

x_train[ordinal_cols] = ordinal.fit_transform(x_train[ordinal_cols])

In [9]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier

estimator = GradientBoostingClassifier(random_state=2512)

# Set the minimum number of features to be selected
min_features_to_select = 1

# Set the cross-validation splitting strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create a RFECV object using the estimator
rfecv = RFECV(estimator, min_features_to_select=min_features_to_select, cv=cv)

# Fit the data
rfecv.fit(x_train, y_sample)

# Get integer index of the features selected
feature_index = rfecv.get_support(indices=True)

# Get a mask of the features selected
feature_mask = rfecv.support_

# Get selected feature names
feature_names = rfecv.get_feature_names_out()

# Get the number of features retained
feature_number = rfecv.n_features_

# Get results
results = pd.DataFrame(rfecv.cv_results_)

# Get RFECV score
rfecv_score = rfecv.score(x_train, y_sample)

# Print feature number, names and score
print("Original feature number:", len(x_train.columns))
print("Optimal feature number:", feature_number)
print("Selected features:", feature_names)
print("Score:", rfecv_score)

Original feature number: 20
Optimal feature number: 18
Selected features: ['CheckingStatus' 'LoanDuration' 'CreditHistory' 'LoanPurpose'
 'LoanAmount' 'ExistingSavings' 'EmploymentDuration' 'InstallmentPercent'
 'Sex' 'OthersOnLoan' 'CurrentResidenceDuration' 'OwnsProperty' 'Age'
 'InstallmentPlans' 'Housing' 'ExistingCreditsCount' 'Job' 'Telephone']
Score: 0.8342085521380345
